In [2]:
import pandas as pd
import numpy as np

In [4]:
# 1. Load
df = pd.read_csv('healthcare_dataset.csv')
print(f"Original: {df.shape}") 

Original: (55500, 15)


In [5]:
# 2. Clean Names - fix the messy case 
df['Name'] = df['Name'].str.title().str.strip()
df['Doctor'] = df['Doctor'].str.title().str.strip()
df['Hospital'] = df['Hospital'].str.strip()


In [6]:
# 3. Standardize text columns
df['Gender'] = df['Gender'].str.title()
df['Medical Condition'] = df['Medical Condition'].str.title()
df['Blood Type'] = df['Blood Type'].str.upper().str.strip()

In [7]:
# 4. Dates
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'], errors='coerce')
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'], errors='coerce')

In [8]:
# 5. Calculate Length of Stay - KEY METRIC
df['LOS_days'] = (df['Discharge Date'] - df['Date of Admission']).dt.days

In [9]:
# 6. Billing Amount - remove $ and convert
df['Billing Amount'] = df['Billing Amount'].astype(str).str.replace('$','').str.replace(',','').astype(float)

In [10]:
# 7. Remove outliers
df = df[(df['Age'] > 0) & (df['Age'] < 100)]
df = df[(df['LOS_days'] > 0) & (df['LOS_days'] < 100)] # remove 200+ day stays

In [11]:
# 8. Business Insights
print("\n=== BUSINESS INSIGHTS ===")
print(f"Total Revenue: ${df['Billing Amount'].sum():,.0f}")
print(f"Avg Billing: ${df['Billing Amount'].mean():,.0f}")
print(f"Avg LOS: {df['LOS_days'].mean():.1f} days (Benchmark 4-6)")
print("\nTop 3 Costly Conditions:")
print(df.groupby('Medical Condition')['Billing Amount'].sum().sort_values(ascending=False).head(3))
print("\nHospital Variance:")
print(f"Max Avg: ${df.groupby('Hospital')['Billing Amount'].mean().max():,.0f}")
print(f"Min Avg: ${df.groupby('Hospital')['Billing Amount'].mean().min():,.0f}")


=== BUSINESS INSIGHTS ===
Total Revenue: $1,417,432,043
Avg Billing: $25,539
Avg LOS: 15.5 days (Benchmark 4-6)

Top 3 Costly Conditions:
Medical Condition
Diabetes     2.385397e+08
Obesity      2.382149e+08
Arthritis    2.373291e+08
Name: Billing Amount, dtype: float64

Hospital Variance:
Max Avg: $52,373
Min Avg: $-2,008


In [12]:
# Fix negative billing (data entry error)
df = df[df['Billing Amount'] > 0]

print(f"After cleaning negatives: {df.shape}")
print(f"New Min Avg: ${df.groupby('Hospital')['Billing Amount'].mean().min():,.0f}")

After cleaning negatives: (55392, 16)
New Min Avg: $9


In [17]:
df = df[df['Billing Amount'] > 1000]  # real hospital bills
print(f"Final shape: {df.shape}")
print(f"Total Revenue: ${df['Billing Amount'].sum():,.0f}")
print(f"Avg Billing: ${df['Billing Amount'].mean():,.0f}")
print(f"Avg LOS: {df['LOS_days'].mean():.1f} days")
print(f"Hospital Min Avg: ${df.groupby('Hospital')['Billing Amount'].mean().min():,.0f}")
print(f"Hospital Max Avg: ${df.groupby('Hospital')['Billing Amount'].mean().max():,.0f}")

Final shape: (55074, 16)
Total Revenue: $1,417,297,313
Avg Billing: $25,734
Avg LOS: 15.5 days
Hospital Min Avg: $1,000
Hospital Max Avg: $52,373


In [19]:
# 9. Save cleaned dataset
df.to_csv('healthcare_clean.csv', index=False)
print("\nSaved healthcare_clean.csv")


Saved healthcare_clean.csv


In [18]:
df.head()


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,LOS_days
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,2
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,15
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal,30
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal,20
